shop

> cart and checkout scaffolding for agents — find products, add them, verify the cart, fill checkout forms

An agent driving a real shopping cart hits the same four walls on every site: it does not know
which CSS selector holds a product card, it cannot tell which search result is the right one, it
guesses at the add-to-cart control, and — worst — it has no way to know whether the click worked,
so it reports success it never observed.

`fossick.shop` removes all four. It runs on the same persistent debug Chrome as
[`cdp_connect`](https://vedicreader.github.io/fossick/cdp.html#cdp_connect), so a logged-in
session, a chosen delivery store and a cookie banner already dismissed by hand all carry over.
Products come back as a numbered list an agent picks from by index or by name, every mutating call
re-reads the cart and returns *what changed*, and nothing is ever driven by a hand-written
selector. There are no per-site scripts to maintain: one generic layer reads any storefront's DOM,
with a deterministic platform API (Shopify's `/cart.js`) used when the site exposes one.

## The in-page layer

Everything that has to read a live DOM happens in one injected script. It tags what it finds with
`data-fk*` attributes, which is how Python gets back a CDP node id to click without either side
inventing a selector: the page finds the element, Python acts on it.

Card boundaries are the hard part. A product "card" is not a class name — it differs on every
site — so the script derives it: group all content links by their normalised URL, then climb from
each link until the parent would swallow a *second* product link. That ancestor is the card, and
price, add button, quantity control and stock status are read from inside it.

The script is written cheap-condition-first throughout, because it runs over every element on the
page and the cart scan runs again every 0.4s while an action is being verified. `innerText`,
`getComputedStyle` and `getBoundingClientRect` all force a layout; `textContent` does not. So each
scan tests its text condition on `textContent` and only then asks whether the element is visible,
and the card climb memoises its link counts. On a 300-product page that is a cart read in 4ms
rather than 17ms, and a product scan in 12ms rather than 44ms.

In [ ]:
#| default_exp shop

In [ ]:
#| export
import re, time, asyncio, difflib, html
from urllib.parse import quote_plus, urlparse
from functools import wraps
from fastcore.all import patch, store_attr, first, L
from fastcdp import *
from fossick.core import syncy
from fossick.cdp import cdp_connect

In [ ]:
#| export
SHOP_JS = r'''
window.__fk = (() => {
const PRICE_RX = /(?:AU\$|NZ\$|US\$|CA\$|\$|€|£|₹|¥)\s?\d[\d,]*(?:\.\d{1,2})?|\d[\d,]*(?:\.\d{1,2})?\s?(?:AUD|USD|EUR|GBP|NZD|CAD)\b/;
const ADD_RX  = /^(?:add|add\s?\+|\+\s?add|add to (?:cart|bag|basket|trolley|order|box)|add item|add one|quick add|quick buy|buy now|order now)\b/i;
const NOTADD_RX = /add to (list|wish|registry|favou?rites|compare)|save for later|remove|delete|view details/i;
const OOS_RX  = /out of stock|sold out|currently unavailable|notify me|back in stock|temporarily out/i;
const TOAST_RX = /added to (your )?(cart|bag|basket|trolley|order)|added!|item added|added successfully|now in your (cart|bag|basket)/i;
const UTIL_RX = /\/(cart|checkout|login|signin|sign-in|register|account|help|wishlist|favourites|compare|quickview|quick-view|size-guide|sizing|reviews?|questions|blog|news|policies|pages|terms|privacy|contact|about|offer-listing|sell|gift)(\/|$|\?)|[?&](sort|order|page|filter|view)=/i;
const KEEP_Q  = /^(variant|variant_id|id|product_id|productid|sku|pid|item|itemid|p|colour|color|size)$/i;
const CARTHREF_RX = /\/(cart|basket|trolley|bag|checkout\/cart|gp\/cart)(\/|$|\?|\.)/i;
const COUNT_RX = /count|badge|bubble|indicator|num|items\b/i;
const CART_RX = /\b(cart|basket|trolley|bag)\b/i;
const SEARCH_SEL = 'input[type=search], [role=searchbox], input[name=q], input[name=s], input[name=search], input[name=searchTerm], input[name=search_query], input[name=query], input[name=keyword], input[name=k], input[placeholder*=search i], input[aria-label*=search i]';

const rect = el => (el && el.getBoundingClientRect ? el.getBoundingClientRect() : {width: 0, height: 0});
const shown = el => {   // laid out at all. Deliberately ignores opacity: grids that fade cards in on
  if (!el) return false; // scroll are still real, and requiring opacity>0 hides most of the page.
  const r = rect(el);
  return (r.width > 0 || r.height > 0) && getComputedStyle(el).visibility !== 'hidden';
};
const txt = el => (el && (el.innerText || el.textContent) || '').replace(/\s+/g, ' ').trim();
const raw = el => (el && el.textContent) || '';   // no layout, unlike innerText — use it to pre-filter
const num = s => {
  const m = String(s == null ? '' : s)
    .replace(/(\d)[ \t\u00a0]+(\d{2})(?!\d)/g, '$1.$2')   // "$27 82" -> "$27.82" (split fraction span)
    .replace(/\s+/g, '').match(PRICE_RX);
  if (!m) return null;
  const n = parseFloat(m[0].replace(/[^\d.,]/g, '').replace(/,(?=\d{3}(\D|$))/g, '').replace(',', '.'));
  return isFinite(n) ? n : null;
};
const priceOf = el => {   // explicit price markup first — amazon splits $12.99 across spans
  for (const sel of ['.a-offscreen', '[itemprop=price]', 'meta[itemprop=price]', '[data-price]', '[class*=price i]']) {
    for (const e of el.querySelectorAll(sel)) {
      const v = num(e.getAttribute('content') || e.getAttribute('data-price') || txt(e));
      if (v) return v;
    }
  }
  return num(txt(el));
};
const norm = href => {   // same-origin URL, tracking params and /ref= segments dropped
  try {
    const u = new URL(href, location.href);
    if (u.origin !== location.origin) return null;
    u.hash = '';
    u.pathname = u.pathname.replace(/\/ref=[^/]*$/, '').replace(/\/+$/, '');
    [...u.searchParams.keys()].forEach(k => { if (!KEEP_Q.test(k)) u.searchParams.delete(k); });
    return u.pathname && u.pathname !== '/' ? u.origin + u.pathname + (u.search || '') : null;
  } catch (e) { return null; }
};
const shape = url => {   // /x/dp/B008S6GEB2 -> /*/dp/* ; /products/291-apples-fuji -> /products/*
  try {
    return new URL(url).pathname.split('/').filter(Boolean)
      .map(s => (/\d/.test(s) || s.length > 24 || s.includes('-') ? '*' : s)).join('/');
  } catch (e) { return ''; }
};
const inChrome = el => !!el.closest('header, nav, footer, [role=navigation], [role=banner], [role=contentinfo]');

const findBtn = card => {
  let loose = null;
  for (const b of card.querySelectorAll('button, input[type=submit], input[type=button], a[href], [role=button]')) {
    if (b.disabled || !shown(b)) continue;
    const label = (b.value || b.getAttribute('aria-label') || txt(b) || b.title || '').trim();
    if (NOTADD_RX.test(label) || /wish|compare|subscribe/i.test(label)) continue;
    if (ADD_RX.test(label)) return b;
    const key = [b.className, b.name, b.id, b.getAttribute('data-testid')].join(' ');
    if (!loose && /add[-_]?to|addto|add[-_]?cart|add[-_]?item/i.test(key) &&
        !/remove|delete|wish|compare/i.test(key)) loose = b;
  }
  return loose;
};
const findQty = card => {
  for (const e of card.querySelectorAll('select, input[type=number], input[name*=quant i], input[name*=qty i], input[aria-label*=quant i]')) {
    if (!shown(e)) continue;
    const key = [e.name, e.id, e.className, e.getAttribute('aria-label'), txt(e.closest('label'))].join(' ');
    if (e.tagName === 'SELECT' && !/qty|quantity/i.test(key)) continue;
    return {el: e, kind: e.tagName === 'SELECT' ? 'select' : 'input'};
  }
  const plus = [...card.querySelectorAll('button, [role=button]')].find(b => shown(b) &&
    /^\+$|increase|increment|add one more|plus/i.test((b.getAttribute('aria-label') || txt(b) || b.className)));
  return plus ? {el: plus, kind: 'stepper'} : null;
};
const titleOf = (card, els) => {
  for (const sel of ['h1', 'h2', 'h3', 'h4', '[itemprop=name]', '[class*=title i]', '[class*=product-name i]', '[class*=name i]']) {
    const t = txt(card.querySelector(sel));
    if (t && t.length > 2 && num(t) === null) return t.slice(0, 160);
  }
  const byText = els.map(e => txt(e)).filter(t => t.length > 2 && num(t) === null).sort((a, b) => b.length - a.length)[0];
  if (byText) return byText.slice(0, 160);
  const byAlt = els.map(e => ((e.querySelector('img') || {}).alt) || e.getAttribute('aria-label') || '')
                   .sort((a, b) => b.length - a.length)[0];
  return (byAlt || '').slice(0, 160);
};
const variantId = card => {   // shopify add-to-cart forms carry the variant id in input[name=id]
  const f = card.querySelector('form[action*="/cart/add"], form[action*="/cart"]');
  const i = (f || card).querySelector('input[name=id]:not([disabled]), select[name=id]');
  return /^\d{6,}$/.test((i && i.value) || '') ? i.value : null;
};

function rawCards(shapeFilter) {
  const groups = new Map(), linkHref = new Map();
  for (const a of document.querySelectorAll('a[href]')) {
    const raw = a.getAttribute('href') || '';
    if (/^(#|javascript:|mailto:|tel:)/.test(raw)) continue;
    const h = norm(raw);
    if (!h || UTIL_RX.test(h) || !shown(a)) continue;
    if (!a.querySelector('img, picture, svg') && txt(a).length < 3) continue;   // icon-only utility links
    if (!groups.has(h)) groups.set(h, []);
    groups.get(h).push(a);
    linkHref.set(a, h);
  }
  // Count only links that passed the filter above. Re-deriving hrefs here would re-admit
  // href="#", which resolves to the current page and can collide with a real candidate —
  // that stops the climb one level short and loses the add button.
  const counted = new Map();   // every card climbs the same ancestors; count each element once
  const nHrefs = el => {
    if (counted.has(el)) return counted.get(el);
    const s = new Set();
    for (const a of el.querySelectorAll('a[href]')) { const h = linkHref.get(a); if (h) s.add(h); }
    counted.set(el, s.size);
    return s.size;
  };
  const seen = new Set(), out = [];
  for (const [h, els] of groups) {
    let card = els[0], up = 0;
    while (card.parentElement && up < 12) {
      const p = card.parentElement;
      if (p === document.body || nHrefs(p) > 1) break;
      card = p; up++;
    }
    if (seen.has(card) || card === document.body) continue;
    const price = priceOf(card), btn = findBtn(card), title = titleOf(card, els);
    if (!title || title.length < 3) continue;
    if (price === null && !btn) continue;
    if (inChrome(card) && !btn) continue;
    seen.add(card);
    out.push({card, url: h, title, price, btn, qty: findQty(card), vid: variantId(card)});
  }
  if (shapeFilter !== false && out.length >= 4) {   // drop stragglers: keep the dominant URL shape
    const tally = {};
    out.forEach(c => { const s = shape(c.url); tally[s] = (tally[s] || 0) + 1; });
    const best = Object.entries(tally).sort((a, b) => b[1] - a[1])[0];
    if (best && best[1] >= out.length / 2) return out.filter(c => shape(c.url) === best[0]);
  }
  return out;
}

function isPdp() {
  if (window.ShopifyAnalytics && ((window.ShopifyAnalytics.meta || {}).page || {}).pageType === 'product') return true;
  const og = document.querySelector('meta[property="og:type"]');
  if (og && /product/i.test(og.content || '')) return true;
  for (const s of document.querySelectorAll('script[type="application/ld+json"]')) {
    try {
      const j = JSON.parse(s.textContent);
      const arr = [].concat(Array.isArray(j) ? j : [j], j['@graph'] || []);
      if (arr.some(o => o && /Product/i.test(String(o['@type'] || '')))) return true;
    } catch (e) {}
  }
  return false;
}
function single() {   // the one buyable thing on a product-detail page
  const btn = findBtn(document.body);
  const h1 = document.querySelector('h1');
  if (!btn && !h1) return [];
  const scope = (h1 && h1.closest('main, [role=main], form, article, section')) || document.body;
  const title = txt(h1) || (document.querySelector('meta[property="og:title"]') || {}).content || document.title;
  return [{card: scope, url: norm(location.href) || location.href, title: (title || '').trim().slice(0, 160),
           price: priceOf(scope), btn, qty: findQty(scope), vid: variantId(scope)}];
}

function tag(list) {
  const ATTRS = ['data-fk', 'data-fk-add', 'data-fk-qty'];
  document.querySelectorAll('[data-fk],[data-fk-add],[data-fk-qty]')
          .forEach(e => ATTRS.forEach(a => e.removeAttribute(a)));
  return list.map((c, i) => {
    c.card.setAttribute('data-fk', i);
    if (c.btn) c.btn.setAttribute('data-fk-add', i);
    if (c.qty) c.qty.el.setAttribute('data-fk-qty', i);
    return {i, title: c.title, price: c.price, url: c.url, add: !!c.btn,
            add_label: c.btn ? (c.btn.value || c.btn.getAttribute('aria-label') || txt(c.btn) || '').trim().slice(0, 40) : null,
            qty: c.qty ? c.qty.kind : null, vid: c.vid, related: c.related || null,
            oos: OOS_RX.test(txt(c.card)) || null};
  });
}
function products(limit) {
  const grid = rawCards();
  let list = grid;
  if (isPdp() || grid.length < 2) {   // main product first, recommendations flagged `related`
    const s = single();
    if (s.length && s[0].title) {
      const rest = grid.filter(c => c.url !== s[0].url);
      rest.forEach(c => { c.related = true; });
      list = s.concat(rest);
    }
  }
  return tag(list.slice(0, limit || 60));
}

function badges() {
  // Three tiers, strongest first: an explicit count node, a link to the cart page, a cart-named
  // control. Arbitrary page text is never trusted — "Filter Basket" in a product title is not a badge.
  const tiers = [[], [], []], seen = new Set();
  const grab = el => {
    const m = [el.getAttribute('aria-label'), el.title, txt(el)].filter(Boolean).join(' ').match(/(\d+)/);
    return m ? parseInt(m[1]) : null;
  };
  const push = (t, el, src) => {
    const n = grab(el);
    if (n === null || n > 999 || seen.has(el) || el.closest('[data-fk]')) return;
    const body = [el.getAttribute('aria-label'), txt(el)].filter(Boolean).join(' ');
    if (PRICE_RX.test(body) && !/\bitems?\b|\bcount\b/i.test(body)) return;   // that is a price
    seen.add(el);
    tiers[t].push({n, tier: t, src: (src || '').replace(/\s+/g, ' ').trim().slice(0, 50)});
  };
  // `shown` costs a getComputedStyle and a getBoundingClientRect, so every loop here tests its
  // cheap textual condition first. This runs on every cart poll, over every element on the page.
  for (const el of document.querySelectorAll('[id],[class],[data-cart-count],[data-testid]')) {
    const key = [el.id, el.className, el.getAttribute('data-testid')].join(' ');
    if (!(/cart|basket|trolley|bag/i.test(key) && COUNT_RX.test(key))) continue;
    if (el.children.length > 2 || /^(INPUT|SELECT|TEXTAREA|OPTION)$/.test(el.tagName) || !shown(el)) continue;
    push(0, el, key);
  }
  for (const a of document.querySelectorAll('a[href]'))
    if (CARTHREF_RX.test(a.getAttribute('href') || '') && shown(a)) push(1, a, txt(a) || a.getAttribute('href'));
  for (const el of document.querySelectorAll('a, button, [role=button], [role=link]')) {
    if (!CART_RX.test(raw(el) + ' ' + (el.getAttribute('aria-label') || '')) || !shown(el)) continue;
    const nm = (el.getAttribute('aria-label') || txt(el) || '').trim();
    if (nm.length > 60 && !CART_RX.test(el.getAttribute('aria-label') || '')) continue;
    if (CART_RX.test(nm)) push(2, el, nm);
  }
  return tiers.find(t => t.length) || [];
}
const TOTAL_RX = /(sub[- ]?total|order total|cart total|estimated total|basket total|trolley total)/i;
function totals() {
  for (const el of document.body.querySelectorAll('*')) {
    const c = raw(el);
    if (c.length > 200 || !TOTAL_RX.test(c) || el.children.length > 3 || !shown(el)) continue;
    const t = txt(el);
    if (t.length > 70) continue;
    const v = num(t) != null ? num(t) : (num(txt(el.nextElementSibling)) != null ? num(txt(el.nextElementSibling)) : num(txt(el.parentElement)));
    if (v) return v;
  }
  return null;
}
const mode = ns => {   // the reading most sources agree on, not the biggest one
  const t = {};
  ns.forEach(n => { t[n] = (t[n] || 0) + 1; });
  return Object.entries(t).sort((a, b) => b[1] - a[1] || b[0] - a[0])[0][0] * 1;
};
function cart(wantLines) {
  const b = badges();
  const out = {count: b.length ? mode(b.map(x => x.n)) : null, subtotal: totals(),
               badges: b.slice(0, 4), url: location.href, title: document.title};
  if (wantLines) out.lines = lines();
  return out;
}
// Everything a verified action needs to know, in one round trip: the cart to compare, the toast
// that may have appeared instead, and the site's own complaint if it refused.
const watch = (scope, wantLines) => ({cart: cart(wantLines), toast: toast(scope), err: err(scope)});
const findRm = row => [...row.querySelectorAll('button, input[type=submit], a, [role=button]')].find(b => shown(b) && (
  /^(remove|delete|trash|✕|×|✖)|remove (item|from)|delete (item|from)/i.test(
    (b.getAttribute('aria-label') || b.value || txt(b) || b.className)) ||
  /remove|delete/i.test((b.closest('td,th') || {}).className || '')));
function lines() {
  const ATTRS = ['data-fk-line', 'data-fk-lineqty', 'data-fk-rm'];
  document.querySelectorAll('[data-fk-line],[data-fk-lineqty],[data-fk-rm]')
          .forEach(e => ATTRS.forEach(a => e.removeAttribute(a)));
  // A cart line is a product card you can change or remove but not add. Scanning for rows that
  // merely contain a price also matches the "customers also bought" carousels on a cart page.
  let rows = rawCards(false).map(c => c.card).filter(r => (findQty(r) || findRm(r)) && !findBtn(r));
  if (!rows.length) rows = [...document.querySelectorAll('tr, li, [class*=line-item i], [class*=lineitem i], [class*=cart-item i], [class*=cart__item i], [class*=basket-item i], [class*=cart_line i]')]
    .filter(r => shown(r) && PRICE_RX.test(txt(r)) && txt(r).length < 800 && (findQty(r) || findRm(r)) &&
                 !r.querySelector('tr, li, [class*=cart-item i], [class*=line-item i]'));
  return rows.map((r, i) => {
    r.setAttribute('data-fk-line', i);
    const q = findQty(r), rm = findRm(r);
    if (q) q.el.setAttribute('data-fk-lineqty', i);
    if (rm) rm.setAttribute('data-fk-rm', i);
    return {i, text: txt(r).slice(0, 120), price: priceOf(r),
            qty: q ? (q.el.value || null) : null, qty_kind: q ? q.kind : null, remove: !!rm};
  });
}
function err(scope) {   // the site's own complaint after a failed action
  const root = (scope == null ? null : document.querySelector('[data-fk="' + scope + '"]')) || document.body;
  const sels = '[role=alert], [class*=error i], [class*=failure i], [class*=warning i], [class*=notice i]';
  for (const el of [...root.querySelectorAll(sels), ...document.querySelectorAll(sels)]) {
    if (!shown(el)) continue;
    const t = txt(el);
    if (t.length > 3 && t.length < 200) return t.slice(0, 160);
  }
  return null;
}
function toast(scope) {   // "Added to cart" style confirmation; compare before/after, it can pre-exist
  const root = (scope == null ? null : document.querySelector('[data-fk="' + scope + '"]')) || document.body;
  for (const el of root.querySelectorAll('*')) {
    if (!TOAST_RX.test(raw(el)) || el.children.length > 2 || !shown(el)) continue;
    const t = txt(el);
    if (t.length < 120 && TOAST_RX.test(t)) return t.slice(0, 80);
  }
  for (const el of document.querySelectorAll('[role=status], [role=alert], [aria-live]')) {
    const t = txt(el);
    if (t && TOAST_RX.test(t)) return t.slice(0, 80);
  }
  return null;
}

function platform() {
  if (window.Shopify || document.querySelector('script[src*="cdn.shopify"], link[href*="cdn.shopify"]')) return 'shopify';
  if (window.wc_add_to_cart_params || /woocommerce/.test(document.body.className)) return 'woocommerce';
  if (window.BCData) return 'bigcommerce';
  if (window.Magento || document.querySelector('script[src*="/mage/"]')) return 'magento';
  if (window.__NEXT_DATA__) return 'next';
  return 'generic';   // never null: shop_prep probes with `__fk ? __fk.platform() : null`
}
const dialogs = () => [...document.querySelectorAll('[role=dialog], [aria-modal=true], dialog[open]')]
  .filter(d => { const r = rect(d); return shown(d) && r.width > 120 && r.height > 60 && txt(d).length > 10; });
function blockers() {
  const out = [], body = txt(document.body).slice(0, 5000), dlg = dialogs();
  const dtxt = dlg.map(txt).join(' ').slice(0, 2000);
  if (/accept (all )?cookies|we use cookies|cookie (policy|settings|preferences)/i.test(body)) out.push('cookie-banner');
  if (/postcode|post code|zip code|choose (a )?store|select (a )?store|set (your )?location|delivery suburb/i.test(dtxt)) out.push('location-required');
  if (/(sign in|log in|login) to (continue|add|shop|order|see prices)|members only/i.test(body) ||
      [...document.querySelectorAll('input[type=password]')].some(shown)) out.push('login-required');
  if (document.querySelector('iframe[src*=recaptcha], iframe[src*=turnstile], iframe[src*=hcaptcha], #px-captcha, #challenge-form')) out.push('captcha');
  if (dlg.length) out.push('modal-open');
  return [...new Set(out)];
}
function dismiss() {
  const RX = /^(accept( all)?( cookies)?|allow all|i agree|agree|got it|ok(ay)?|close|no thanks|not now|dismiss|reject all|decline|continue without)\b/i;
  const hit = [];
  const scopes = [...dialogs(), ...document.querySelectorAll('[id*=cookie i],[class*=cookie i],[class*=consent i],[id*=consent i],[class*=gdpr i]')].filter(shown);
  for (const s of scopes.slice(0, 8)) {
    const b = [...s.querySelectorAll('button, [role=button], a')].find(x => shown(x) && RX.test((x.getAttribute('aria-label') || txt(x)).trim()));
    if (b) { hit.push(((b.getAttribute('aria-label') || txt(b)) || '').trim().slice(0, 40)); b.click(); }
  }
  return hit;
}

function labelOf(el) {
  let l = el.id ? document.querySelector('label[for="' + CSS.escape(el.id) + '"]') : null;
  if (!l) l = el.closest('label');
  if (!l && el.getAttribute('aria-labelledby')) l = document.getElementById(el.getAttribute('aria-labelledby').split(' ')[0]);
  return (l ? txt(l).slice(0, 80) : '') || el.getAttribute('aria-label') || el.placeholder || '';
}
function fields() {
  document.querySelectorAll('[data-fk-f]').forEach(e => e.removeAttribute('data-fk-f'));
  const out = [];
  let i = 0;
  for (const el of document.querySelectorAll('input, select, textarea')) {
    if (!shown(el) || ['hidden', 'submit', 'button', 'image', 'reset'].includes(el.type)) continue;
    el.setAttribute('data-fk-f', i);
    out.push({i: i++, tag: el.tagName.toLowerCase(), type: el.type || null, name: el.name || null,
              id: el.id || null, autocomplete: el.getAttribute('autocomplete') || null, label: labelOf(el),
              required: !!el.required, value: el.type === 'password' ? null : (el.value || ''),
              options: el.tagName === 'SELECT' ? [...el.options].slice(0, 80).map(o => ({v: o.value, t: txt(o)})) : null});
  }
  return out;
}
function setVal(el, v) {   // native setter + events, so React/Vue controlled inputs actually update
  const P = el.tagName === 'SELECT' ? HTMLSelectElement : el.tagName === 'TEXTAREA' ? HTMLTextAreaElement : HTMLInputElement;
  try { el.focus(); } catch (e) {}
  if (el.type === 'checkbox' || el.type === 'radio') { if (el.checked !== !!v) el.click(); return el.checked; }
  Object.getOwnPropertyDescriptor(P.prototype, 'value').set.call(el, v);
  ['input', 'change', 'blur'].forEach(t => el.dispatchEvent(new Event(t, {bubbles: true})));
  return el.value;
}
function pick(el, v) {
  const w = String(v).toLowerCase().trim();
  const o = [...el.options].find(o => o.value.toLowerCase() === w || txt(o).toLowerCase() === w) ||
            [...el.options].find(o => txt(o).toLowerCase().includes(w) || (o.value || '').toLowerCase().includes(w));
  return o ? setVal(el, o.value) : null;
}
function fill(idx, v) {
  const el = document.querySelector('[data-fk-f="' + idx + '"]');
  if (!el) return null;
  return el.tagName === 'SELECT' ? pick(el, v) : setVal(el, v);
}
function setQty(attr, i, n) {
  const el = document.querySelector('[' + attr + '="' + i + '"]');
  if (!el) return null;
  if (el.tagName === 'SELECT') return pick(el, String(n));
  return setVal(el, String(n));
}
function searchBox(q) {
  const el = [...document.querySelectorAll(SEARCH_SEL)].find(shown);
  if (!el) return null;
  el.setAttribute('data-fk-search', '1');
  if (q != null) setVal(el, q);
  return {name: el.name || null, form: !!el.form, id: el.id || null};
}
function submitSearch() {
  const el = document.querySelector('[data-fk-search]');
  if (!el) return null;
  if (el.form) { el.form.submit(); return 'form'; }
  return null;
}
async function variants() {
  const m = (window.ShopifyAnalytics && window.ShopifyAnalytics.meta && window.ShopifyAnalytics.meta.product) ||
            (window.meta && window.meta.product);
  let out = m && m.variants
    ? m.variants.map(v => ({id: String(v.id), name: v.name || v.public_title || '', price: v.price / 100, available: null}))
    : null;
  const h = location.pathname.match(/\/products\/([^/?#]+)/);
  if (h) try {                       // /products/<handle>.js is the only source with stock per variant
    const r = await fetch('/products/' + h[1] + '.js', {headers: {Accept: 'application/json'}});
    if (r.ok) {
      const p = await r.json();
      out = p.variants.map(v => ({id: String(v.id), name: v.public_title || v.title || '', price: v.price / 100,
                                  available: !!v.available}));
    }
  } catch (e) {}
  return out;
}
return {products, cart, lines, watch, toast, err, platform, blockers, dismiss, fields, fill, setQty,
        searchBox, submitSearch, variants, price: num, cartLink: () => {
          const a = [...document.querySelectorAll('a[href]')].find(a => CARTHREF_RX.test(a.getAttribute('href') || ''));
          return a ? new URL(a.getAttribute('href'), location.href).href : null;
        }};
})();
'__fk ready'
'''

## Site hints

The generic layer is the whole product; these are hints, not drivers — a search URL saves a
round-trip through the site's own search box, and a cart URL saves guessing. A site absent from
this table works exactly the same way, it just types into the search box it finds.

The two Australian supermarkets are listed from their public URL patterns but could not be
exercised from a datacentre IP (Coles serves an empty JS shell, Woolworths answers 403). Both are
precisely the case `session=`-style automation exists for: run them through a debug Chrome on a
real machine, where the store and login are already set.

In [ ]:
#| export
SITES = {
    'coles.com.au':         dict(search='https://www.coles.com.au/search/products?q={q}', cart='https://www.coles.com.au/trolley',
                                 note='pick a store first; bot-walled from datacentre IPs'),
    'woolworths.com.au':    dict(search='https://www.woolworths.com.au/shop/search/products?searchTerm={q}',
                                 cart='https://www.woolworths.com.au/shop/mytrolley',
                                 note='set a delivery suburb first; bot-walled from datacentre IPs'),
    'amazon.':              dict(search='https://{host}/s?k={q}', cart='https://{host}/gp/cart/view.html'),
    'ceresfairfood.org.au': dict(search='https://members.ceresfairfood.org.au/products/search?q={q}',
                                 cart='https://members.ceresfairfood.org.au/cart', note='members shop: log in to add'),
}

def site_hint(url:str) -> dict:
    "Hints (`search`, `cart`, `note`) for the host of `url`; `{}` when the site is not listed."
    host = urlparse(url).netloc.lower()
    for k, v in SITES.items():
        if k in host: return {kk: (vv.replace('{host}', host) if isinstance(vv, str) else vv) for kk, vv in v.items()}
    return {}

## Reading a page

`shop_prep` injects the script if it is not already there — including after a navigation, which
wipes it — so every other call can assume it is present. `platform()` answers `'generic'` rather
than nothing when it recognises no platform, which makes `window.__fk ? __fk.platform() : null` a
single round trip that both probes and reports.

Everything below evaluates JavaScript through `page.eval`/`page.eval_json` from
[`fossick.cdp`](https://vedicreader.github.io/fossick/cdp.html), which await promises, decode
objects and raise `JSError` on a page exception — so there is no private JS bridge in this
module.

In [ ]:
#| export
class ShopError(RuntimeError):
    "A shop action could not be carried out (bad index, no add control, nothing to match)."
    pass

def _origin(url:str) -> str:
    "Scheme and host of `url`, with no path."
    return f'{(u:=urlparse(url)).scheme}://{u.netloc}'

@patch
async def shop_prep(page:Page, force:bool=False) -> str:
    """The platform this store runs on — `shopify`, `woocommerce`, `bigcommerce`, `magento`, `next`
    or `generic` — injecting the in-page helpers first if the page has none (a navigation wipes them)."""
    if not force and (plat := await page.eval('window.__fk ? __fk.platform() : null')): return plat
    await page.eval(SHOP_JS)
    return await page.eval('__fk.platform()')

@patch
async def shop_products(page:Page, limit:int=40) -> list:
    "Products on the current page as `[{i, title, price, url, add, qty, oos, vid}]`. `i` feeds `shop_add`."
    await page.shop_prep()
    return await page.eval_json('__fk.products', limit) or []

#: Shopify's own cart, read through the API it ships rather than off the page
_CART_FN = ("async () => {const r = await fetch('/cart.js', {headers:{Accept:'application/json'}});"
            "if (!r.ok) return null; const c = await r.json();"
            "return {count:c.item_count, subtotal:c.total_price/100, currency:c.currency, url:location.href,"
            "lines:c.items.map((it,n)=>({i:n, title:it.title, qty:it.quantity,"
            "price:it.final_line_price/100, key:it.key}))};}")

@patch
async def shop_cart(page:Page, lines:bool=False) -> dict:
    "Current cart: `{count, subtotal, source}` (+ `lines` if asked). Uses Shopify's `/cart.js` when available."
    if await page.shop_prep() == 'shopify' and (c := await page.eval_json(_CART_FN)):
        for l in c['lines']: l['title'] = html.unescape(l['title'] or '')   # cart.js escapes it
        return dict(c, source='shopify/cart.js')
    return dict(await page.eval_json('__fk.cart', lines), source='dom')

@patch
async def shop_lines(page:Page) -> list:
    "Cart line items read from the page: `[{i, text, price, qty, remove}]`. Run this on the cart page."
    await page.shop_prep()
    return await page.eval('__fk.lines()') or []

@patch
async def shop_blockers(page:Page) -> list:
    "What is standing in the way: `cookie-banner`, `location-required`, `login-required`, `captcha`, `modal-open`."
    await page.shop_prep()
    return await page.eval('__fk.blockers()') or []

@patch
async def shop_dismiss(page:Page) -> list:
    "Click through cookie/consent banners and closable modals. Returns the labels it clicked."
    await page.shop_prep()
    return await page.eval('__fk.dismiss()') or []

## Verified actions

Every mutating call answers the question an agent cannot answer for itself: *did that work?* The
cart is read before the click and polled after it, and the result names the evidence — a changed
item count, a changed subtotal, a new line, or a confirmation toast that was not there before.
When no signal exists at all (a storefront with no cart badge, logged out) the result is
`ok=None`, never `True`.

`__fk.watch` returns the cart, the toast and the site's own error message together, so a baseline
or a poll tick costs one round trip rather than three.

In [ ]:
#| export
def _sig(c:dict) -> tuple:
    "Comparable fingerprint of the line items in a cart reading."
    return tuple(sorted((l.get('title') or l.get('text') or '', l.get('qty')) for l in (c.get('lines') or [])))

def _changed(before:dict, after:dict) -> str:
    "Which cart signal moved between two readings, or '' if none did."
    if before.get('count') != after.get('count') and after.get('count') is not None: return 'count'
    if before.get('subtotal') != after.get('subtotal') and after.get('subtotal') is not None: return 'subtotal'
    return 'lines' if _sig(before) != _sig(after) else ''

async def _baseline(page, scope=None, lines=False) -> tuple:
    "Cart, toast and page error as they stand before an action — `__fk.watch` reads all three in one call."
    st = await page.eval_json('__fk.watch', scope, lines)
    return dict(st['cart'], source='dom'), st['toast'], st['err']

async def _watch(page, before, tout=12, scope=None, toast0=None, lines=False) -> tuple:
    "Poll until a cart signal moves; a confirmation toast that was not there before also counts."
    end, after = time.time() + tout, before
    while time.time() < end:
        await asyncio.sleep(0.4)
        try: after, toast, _ = await _baseline(page, scope, lines)
        except Exception:
            try: await page.shop_prep(force=True)   # the click navigated and took the helpers with it
            except Exception: pass
            continue
        if (how := _changed(before, after)): return True, after, how
        if toast and toast != toast0: return True, after, f'toast: {toast}'
    blind = before.get('count') is None and before.get('subtotal') is None
    return (None if blind else False), after, ''

def _match(items:list, want) -> dict:
    "Resolve `want` (index or name) against `items`, raising with the real options rather than guessing."
    if isinstance(want, int) or str(want).isdigit():
        if (hit := first(p for p in items if p['i'] == int(want))) is None:
            raise ShopError(f'no product #{want}; page has {len(items)}')
        return hit
    w = str(want).lower()
    titles = L(items).attrgot('title').map(lambda t: t or '')
    pool = ([p for p, t in zip(items, titles) if w == t.lower()] or
            [p for p, t in zip(items, titles) if w in t.lower()])
    if not pool:
        close = difflib.get_close_matches(w, titles, n=3, cutoff=0.5)
        raise ShopError(f'no product matching {want!r}. Closest: {close or "none"}. Titles: {titles[:12]}')
    return sorted(pool, key=lambda p: (bool(p.get('related')), len(p['title'] or '')))[0]

In [ ]:
#| export
#: Shopify adds and quantity changes go through its cart API — no clicking, no guessing
_ADD_FN = ("async (id, q) => {const r = await fetch('/cart/add.js', {method:'POST',"
           "headers:{'Content-Type':'application/json', Accept:'application/json'},"
           "body:JSON.stringify({items:[{id:id, quantity:q}]})});"
           "return {status:r.status, body:(await r.text()).slice(0,300)};}")

async def _variant_id(page, it:dict, variant:str) -> tuple:
    "The variant to add — the one on the card, the only one in stock, or the one `variant` names — and the list."
    if it.get('vid'): return it['vid'], None
    if it['url'] != await page.eval('location.href'): await _goto(page, it['url'])
    vs = await page.eval('__fk.variants()') or []
    ok = [v for v in vs if v.get('available') is not False]
    if len(ok) == 1: return ok[0]['id'], vs
    hit = first(v for v in (ok or vs) if variant and str(variant).lower() in (v['name'] or '').lower())
    return (hit['id'] if hit else None), vs

async def _shopify_add(page, it:dict, qty:int, variant:str):
    "Add via `/cart/add.js`. Returns the result, or None when this product cannot be added that way."
    vid, vs = await _variant_id(page, it, variant)
    if vid is None:
        if not vs: return None                       # not a Shopify product page after all: go and click
        ok_vs = [v for v in vs if v.get('available') is not False]
        return dict(ok=False, need='variant', item=it, variants=ok_vs, sold_out=len(vs) - len(ok_vs),
                    error=(f'no in-stock variant matching {variant!r}' if variant else
                           'pick one of `variants` (in stock only): shop_add(item, variant="...")'))
    before = await page.shop_cart(lines=True)
    r = await page.eval_json(_ADD_FN, str(vid), int(qty))
    after = await page.shop_cart(lines=True)
    how = _changed(before, after)
    return dict(ok=bool(how), how=how or f'cart/add.js -> {r.get("status")}', item=it, qty=qty,
                variant_id=str(vid), before=before, after=after, error=None if how else r.get('body'))

@patch
async def shop_add(page:Page, item, qty:int=1, variant:str=None, open_product:bool=True, tout:int=12) -> dict:
    """Add `item` (index from `shop_products`, or a title) to the cart and verify it landed.

    Returns `{ok, item, how, before, after}`. `ok=True` only when a cart signal actually moved;
    `ok=None` means the site exposes no signal to check, and `ok=False` means nothing changed.
    On a Shopify site this posts to `/cart/add.js`, which needs no clicking and no guessing."""
    plat = await page.shop_prep()
    items = await page.shop_products()
    if not items: return dict(ok=False, error='no products found on this page', url=await page.eval('location.href'))
    it = _match(items, item)
    if plat == 'shopify' and (res := await _shopify_add(page, it, qty, variant)) is not None: return res

    if not it['add'] and open_product and it['url'] != await page.eval('location.href'):
        await _goto(page, it['url'])
        if (items := await page.shop_products()):
            try: it = _match(items, it['title'])
            except ShopError: it = items[0]          # the product page's own product is always index 0
    if not it['add']:
        return dict(ok=False, error='no add-to-cart control on this product', item=it,
                    hint='the site may need a size/variant chosen first — see shop_fields()')

    before, toast0, err0 = await _baseline(page, it['i'])
    clicks, set_to = 1, None
    if qty > 1:
        if it['qty'] in ('select', 'input'):
            set_to = await page.eval_json('__fk.setQty', 'data-fk-qty', it['i'], int(qty))
        if set_to is None: clicks = qty              # no usable qty control: click add qty times
    bid = await page.node_for(f'[data-fk-add="{it["i"]}"]')
    if bid is None: return dict(ok=False, error='add control vanished before the click', item=it)
    for n in range(clicks):
        await page.click_settle(bid, timeout=6)
        if n + 1 < clicks:
            await asyncio.sleep(0.6)
            await page.shop_prep()
            bid = await page.node_for(f'[data-fk-add="{it["i"]}"]') or bid
    ok, after, how = await _watch(page, before, tout=tout, scope=it['i'], toast0=toast0)
    out = dict(ok=ok, how=how or None, item=it, qty=qty, clicks=clicks, qty_set=set_to,
               before=before, after=after)
    if not ok:
        e = await page.eval_json('__fk.err', it['i'])
        out['page_error'] = e if e != err0 else None
        out['blockers'] = await page.shop_blockers()
        out['error'] = out['page_error'] or ('no cart signal on this page to verify against'
                       if ok is None else 'cart did not change after the click')
    return out

In [ ]:
#| export
#: set one Shopify line to a new quantity (0 removes it)
_CHANGE_FN = ("async (id, q) => {const r = await fetch('/cart/change.js', {method:'POST',"
              "headers:{'Content-Type':'application/json'}, body:JSON.stringify({id:id, quantity:q})});"
              "return r.status;}")

async def _goto(page, url, tout:int=25):
    "Navigate and tolerate a chatty page that never goes network-idle; re-injects the helpers."
    try: await page.goto(url, timeout=tout)
    except Exception: pass
    await asyncio.sleep(0.3)
    await page.shop_prep(force=True)
    return await page.eval('location.href')

@patch
async def shop_cart_page(page:Page, url:str=None) -> dict:
    "Open the cart page (hint, on-page cart link, then `/cart`) and return the cart with its lines."
    plat, cur = await page.shop_prep(), await page.eval('location.href')
    url = url or site_hint(cur).get('cart') or (f'{_origin(cur)}/cart' if plat == 'shopify' else None) \
              or await page.eval('__fk.cartLink()') or f'{_origin(cur)}/cart'
    await _goto(page, url)
    return await page.shop_cart(lines=True)

def _line(cart:dict, line) -> dict:
    "Cart line `line` from a reading taken with `lines=True`, or a `ShopError` naming what is there."
    ls = cart.get('lines') or []
    if (ln := first(l for l in ls if l['i'] == int(line))) is None:
        raise ShopError(f'no cart line #{line}; cart page shows {len(ls)}')
    return ln

@patch
async def shop_qty(page:Page, line, qty:int, tout:int=12) -> dict:
    "Set the quantity of cart line `line` (index from `shop_lines`, or a Shopify line key). Verified."
    if await page.shop_prep() == 'shopify':
        before = await page.shop_cart(lines=True)
        key = line if isinstance(line, str) and not line.isdigit() else \
              first(l['key'] for l in (before.get('lines') or []) if l['i'] == int(line))
        if key:
            await page.eval_json(_CHANGE_FN, key, int(qty))
            after = await page.shop_cart(lines=True)
            how = _changed(before, after)
            return dict(ok=bool(how), how=how or None, line=line, qty=qty, before=before, after=after)
    before, toast0, _ = await _baseline(page, lines=True)          # the reading also tags the lines
    ln = _line(before, line)
    if not ln['qty_kind']: return dict(ok=False, error='no quantity control on this line', line=ln)
    await page.eval_json('__fk.setQty', 'data-fk-lineqty', int(line), int(qty))
    ok, after, how = await _watch(page, before, tout=tout, toast0=toast0, lines=True)
    return dict(ok=ok, how=how or None, line=ln, qty=qty, before=before, after=after)

@patch
async def shop_remove(page:Page, line, tout:int=12) -> dict:
    "Remove cart line `line`. Verified the same way as `shop_add`."
    if await page.shop_prep() == 'shopify': return await page.shop_qty(line, 0, tout=tout)
    before, toast0, _ = await _baseline(page, lines=True)
    ln = _line(before, line)
    bid = await page.node_for(f'[data-fk-rm="{int(line)}"]')
    if bid is None: return dict(ok=False, error='no remove control on this line', line=ln)
    await page.click_settle(bid, timeout=6)
    ok, after, how = await _watch(page, before, tout=tout, toast0=toast0, lines=True)
    return dict(ok=ok, how=how or None, line=ln, before=before, after=after)

## Checkout forms

Checkout is where guessing hurts most: the same field is "Suburb" in Melbourne, "City" in London
and "Town" in Dublin. `shop_fields` hands back the ground truth — every visible input with its
label, `autocomplete` token, current value and, for a `<select>`, its actual options — so an agent
fills what is on the page instead of what it expects.

`shop_fill` maps a plain profile dict onto those fields, preferring the
[HTML autocomplete token](https://html.spec.whatwg.org/multipage/form-control-infrastructure.html#autofill)
because it is a spec rather than a guess, then the field name, then label synonyms. It re-reads
every field afterwards and reports which values actually stuck. A numeric key sets that
`shop_fields` index directly, so a size or a delivery window goes in the same call as the address.

It will not press a payment button. `submit=` clicks a named button, but anything that reads like
paying needs `confirm=True` on top — an agent that wanders into a checkout cannot spend money by
accident.

In [ ]:
#| export
#: canonical profile key -> (autocomplete tokens, name/id fragments, label synonyms)
FIELD_MAP = {
    'first_name': (('given-name',), ('firstname', 'first_name', 'fname', 'first-name', 'givenname'),
                   ('first name', 'given name', 'forename')),
    'last_name':  (('family-name',), ('lastname', 'last_name', 'lname', 'last-name', 'surname', 'familyname'),
                   ('last name', 'family name', 'surname')),
    'name':       (('name',), ('fullname', 'full_name', 'yourname'), ('full name', 'your name', 'name on')),
    'email':      (('email',), ('email', 'e-mail'), ('email', 'e-mail')),
    'phone':      (('tel', 'tel-national'), ('phone', 'mobile', 'tel', 'contactnumber'),
                   ('phone', 'mobile', 'telephone', 'contact number')),
    'company':    (('organization',), ('company', 'organisation', 'organization', 'business'),
                   ('company', 'organisation', 'business name')),
    'address1':   (('address-line1', 'street-address'), ('address1', 'address_1', 'street', 'addressline1', 'address'),
                   ('address', 'street address', 'address line 1')),
    'address2':   (('address-line2',), ('address2', 'address_2', 'addressline2', 'apartment', 'unit'),
                   ('address line 2', 'apartment', 'unit', 'suite')),
    'city':       (('address-level2',), ('city', 'suburb', 'town', 'locality'), ('city', 'suburb', 'town')),
    'state':      (('address-level1',), ('state', 'province', 'region', 'county'),
                   ('state', 'province', 'region', 'territory', 'county')),
    'postcode':   (('postal-code',), ('postcode', 'postal', 'zip', 'zipcode', 'post_code'),
                   ('postcode', 'postal code', 'zip', 'post code')),
    'country':    (('country', 'country-name'), ('country',), ('country',)),
    'card_name':  (('cc-name',), ('ccname', 'cardholder', 'card_name', 'nameoncard'), ('name on card', 'cardholder')),
    'card_number': (('cc-number',), ('cardnumber', 'ccnumber', 'card_number'), ('card number', 'credit card')),
    'card_exp':   (('cc-exp',), ('expiry', 'exp-date', 'cardexpiry'), ('expiry', 'expiration')),
    'card_month': (('cc-exp-month',), ('expmonth', 'exp_month'), ('expiry month', 'month')),
    'card_year':  (('cc-exp-year',), ('expyear', 'exp_year'), ('expiry year', 'year')),
    'card_cvc':   (('cc-csc',), ('cvc', 'cvv', 'csc', 'securitycode'), ('cvc', 'cvv', 'security code')),
    'notes':      ((), ('note', 'comment', 'instruction', 'message'),
                   ('note', 'comment', 'delivery instruction', 'message')),
}
PAY_RX = re.compile(r'\b(pay|payment|place order|complete order|submit order|confirm (order|purchase|payment)'
                    r'|buy now|purchase)\b', re.I)
_ALNUM = re.compile(r'[^a-z0-9]')
_squash = lambda *ss: _ALNUM.sub('', ''.join(s or '' for s in ss).lower())

def _score(f:dict, key:str) -> int:
    "How well form field `f` matches canonical profile key `key` (0 = not at all)."
    ac, names, labels = FIELD_MAP[key]
    tok = [t for t in (f.get('autocomplete') or '').lower().split()
           if t not in ('on', 'off', 'shipping', 'billing')]
    if tok and any(t in ac for t in tok): return 100
    if tok and ac and tok[-1] not in ac: return 0     # a real, different autocomplete token: not this field
    nm = _squash(f.get('name'), f.get('id'))
    if any(_squash(n) in nm for n in names if n): return 60
    lb = (f.get('label') or '').lower()
    return 40 if lb and any(s in lb for s in labels) else 0

def match_fields(fields:list, profile:dict) -> dict:
    "Best field for each profile key: `{key: field}`. One field per key, highest score wins."
    out, used = {}, set()
    pairs = sorted(((_score(f, k), len(f.get('label') or ''), k, f) for k in profile if k in FIELD_MAP
                    for f in fields), key=lambda t: (-t[0], t[1]))
    for sc, _, k, f in pairs:
        if sc == 0 or k in out or f['i'] in used: continue
        if k.startswith('card_') and not (f.get('type') in ('text', 'tel', 'number', None) or f['tag'] == 'select'): continue
        out[k] = f
        used.add(f['i'])
    return out

In [ ]:
#| export
@patch
async def shop_fields(page:Page) -> list:
    "Every visible form field with its label, `autocomplete` token, value and `<select>` options."
    await page.shop_prep()
    return await page.eval('__fk.fields()') or []

@patch
async def shop_set(page:Page, i:int, value) -> dict:
    "Set form field `i` (index from `shop_fields`) to `value`; returns what the field holds afterwards."
    await page.shop_prep()
    got = await page.eval_json('__fk.fill', int(i), value)
    return dict(i=i, wanted=value, got=got, ok=got is not None)

# A <select> holds the option's value ('VIC') where the profile said 'Victoria', and inputs with a
# mask reformat what they were given — so compare against what the fill returned, loosely.
_same = lambda a, b: _ALNUM.sub('', str(a).lower()) == _ALNUM.sub('', str(b).lower())

@patch
async def shop_fill(page:Page, profile:dict, submit:str=None, confirm:bool=False) -> dict:
    """Fill a checkout form from a profile dict, then re-read the form to confirm what stuck.

    Keys are canonical (`first_name, last_name, email, phone, address1, address2, city, state,
    postcode, country, company, notes, card_*`) and are matched to fields by `autocomplete` token
    first; a numeric key sets that `shop_fields` index directly, for the options a profile has no
    name for (size, colour, delivery window). Returns `{filled, failed, unmatched, fields}`.
    `submit=` clicks that button by name; a payment-looking button additionally needs `confirm=True`."""
    if submit and PAY_RX.search(submit) and not confirm:
        raise ShopError(f'{submit!r} looks like it completes a payment — pass confirm=True to click it')
    fields = await page.shop_fields()
    by_i = {f['i']: f for f in fields}
    hit = {k: by_i[int(k)] for k in profile if str(k).isdigit() and int(k) in by_i}
    hit |= match_fields(fields, {k: v for k, v in profile.items() if not str(k).isdigit()})
    filled, failed = {}, {}
    for k, f in hit.items():
        got = await page.eval_json('__fk.fill', f['i'], profile[k])
        (filled if got is not None else failed)[k] = dict(i=f['i'], label=f['label'] or f['name'], got=got)
    after = {f['i']: f for f in await page.shop_fields()}
    for k, v in filled.items():
        cur = (after.get(v['i']) or {}).get('value')
        v['confirmed'] = bool(cur) and (_same(cur, v['got']) or _same(cur, profile[k])
                                        or str(profile[k]).lower()[:20] in str(cur).lower())
    out = dict(filled=filled, failed=failed, unmatched=[k for k in profile if k not in hit],
               fields=[f for f in fields if f['i'] not in {v['i'] for v in hit.values()}])
    if submit:
        tree = await page.ax_tree()
        bid = tree.find_id(role='button', name=submit) or tree.find_id(role='link', name=submit)
        if bid is None: raise ShopError(f'no button/link named {submit!r}')
        await page.click_settle(bid)
        out |= dict(submitted=submit, url=await page.eval('location.href'))
    return out

## `Shop`: the whole thing, synchronously

`Shop` is the surface an agent should actually use. It is plain synchronous Python over the
persistent browser, so each call is one tool call with a small JSON result — no event loop, no
node ids, no selectors. The methods are not hand-written: each `page.shop_*` coroutine is wrapped
once, keeping its own docstring and signature, so the sync surface cannot drift from the async one.

`shop()` reuses a tab already open on that site instead of opening another, which is what lets a
CLI or MCP call pick up where the last one left off — `--search` then `--add 0` acts on the
results page, not the front door.

```python
s = shop('https://members.ceresfairfood.org.au')
s.search('apples')                       # numbered candidates with prices
s.add('Apples Fuji Organic 500g', qty=2) # -> {'ok': True, 'how': 'count', ...}
s.cart()                                 # -> {'count': 2, 'subtotal': 13.0, ...}
```

In [ ]:
#| export
@patch
async def shop_typed_search(page:Page, q:str):
    "Type `q` into whatever search box the page has and submit it (no search URL needed)."
    await page.shop_prep()
    if not await page.eval_json('__fk.searchBox', q):
        raise ShopError('no search box found on this page — pass a search URL to goto() instead')
    if (bid := await page.node_for('[data-fk-search]')) is not None:
        await page.DOM.focus(backendNodeId=bid)
        for t, k in (('keyDown', 'Enter'), ('keyUp', 'Enter')):
            await page.input.dispatchKeyEvent(type=t, key=k, code=k, windowsVirtualKeyCode=13,
                                              nativeVirtualKeyCode=13, text='\r' if t == 'keyDown' else '')
    await asyncio.sleep(1.2)
    gone = not await page.eval('!!(window.__fk && window.__fk.products)')   # navigated away: Enter worked
    if not gone and not await page.eval('__fk.products(3)'):
        await page.eval('__fk.submitSearch()')                              # no results: submit the form itself
        await asyncio.sleep(1.2)
    await page.shop_prep(force=True)
    return await page.eval('location.href')

#: agent-facing name -> the `Page` coroutine behind it. `shop_*` methods and the two page readers
#: from `fossick.cdp` all become plain synchronous `Shop` methods, docstrings and all.
_SHOP_METHS = dict(platform='shop_prep', products='shop_products', cart='shop_cart',
                   cart_page='shop_cart_page', lines='shop_lines', add='shop_add', set_qty='shop_qty',
                   remove='shop_remove', fields='shop_fields', set='shop_set', fill='shop_fill',
                   blockers='shop_blockers', dismiss='shop_dismiss', snapshot='snapshot', md='md')

def _sync(name:str):
    "The `Page` coroutine `name`, as a `Shop` method that runs it on the shared background loop."
    f = getattr(Page, name)
    @wraps(f)
    def _g(self, *args, **kw): return syncy(f(self.page, *args, **kw), tout=self.tout)
    return _g

class Shop:
    """A shopping session on the persistent debug Chrome: search, add, verify, check out.

    Every `page.shop_*` coroutine is here as a plain synchronous method — one call in, one small
    JSON result out, no event loop and no node ids."""
    def __init__(self, page, port:int=9223, tout:int=180): store_attr()

    @property
    def url(self) -> str: return syncy(self.page.eval('location.href'), tout=self.tout)

    def __repr__(self):
        try: c = self.cart()
        except Exception: return f'<Shop {self.url}>'
        return f"<Shop {self.url} | cart: {c.get('count')} items, subtotal {c.get('subtotal')}>"

    def goto(self, url:str) -> str:
        "Navigate to `url`."
        return syncy(_goto(self.page, url), tout=self.tout)

    def search(self, q:str, limit:int=40) -> list:
        "Search the store for `q` (its known search URL, else its own search box) and list products."
        if (u := site_hint(self.url).get('search')): self.goto(u.format(q=quote_plus(q)))
        elif self.platform() == 'shopify': self.goto(f'{_origin(self.url)}/search?q={quote_plus(q)}')
        else: syncy(self.page.shop_typed_search(q), tout=self.tout)
        return self.products(limit)

    def open(self, url:str) -> str:
        "Point the session at `url`, unless it is already there."
        if self.url != url: self.goto(url)
        return self.url

    def find(self, q:str, limit:int=40) -> list:
        "Products on the current page whose title contains `q`."
        return [p for p in self.products(limit) if q.lower() in (p['title'] or '').lower()]

for _name, _meth in _SHOP_METHS.items(): setattr(Shop, _name, _sync(_meth))

async def _prepared_tab(cdp, url:str):
    "A tab fossick itself left open on this site, if there is one — never a human's unrelated tab."
    for t in await cdp.pages:
        if not (t.get('url') or '').startswith(_origin(url)): continue
        pg = await Page.new(t['targetId'], cdp)
        if await pg.eval('!!window.__fk'): return pg, t['url']
        await cdp.target.detachFromTarget(sessionId=pg.sid)
    return None, None

def shop(url:str=None, port:int=9223, headless:bool=None, extra_flags=None,
         resume:bool=False, tout:int=180) -> Shop:
    """Open a `Shop` on the persistent debug Chrome (starting one if needed), optionally at `url`.

    A tab an earlier `shop()` left open on that site is picked up again rather than piling up
    another one, and by default it is navigated to `url`. `resume=True` leaves it exactly where it
    is instead — which is how the CLI carries a page across invocations, so `--add 0` acts on the
    results `--search` left on screen."""
    async def _open():
        cdp = await cdp_connect(port=port, headless=bool(headless), extra_flags=extra_flags)
        pg, at = await _prepared_tab(cdp, url) if url else (None, None)
        if pg is None: pg = await cdp.new_page()
        if url and at != url and not (resume and at): await _goto(pg, url)
        else: await pg.shop_prep(force=True)
        return pg
    return Shop(syncy(_open(), tout=tout), port=port, tout=tout)

## Tests

The matching and verification logic is pure Python and runs in CI. Anything that needs a browser
is marked `eval: false` — it needs a debug Chrome, and the live examples need the internet.

In [ ]:
assert site_hint('https://www.coles.com.au/search/products?q=milk')['cart'].endswith('/trolley')
assert site_hint('https://www.amazon.com.au/s?k=x')['search'] == 'https://www.amazon.com.au/s?k={q}'
assert site_hint('https://members.ceresfairfood.org.au/products/search')['note']
assert site_hint('https://some-random-shopify-store.com') == {}     # generic path, no hints needed

In [ ]:
#: a checkout form in the shape real ones come in: AU wording, mixed autocomplete coverage, decoys
FORM = [
    dict(i=0, tag='input', type='search', name='q', id='', autocomplete=None, label='Search parts', options=None),
    dict(i=1, tag='input', type='email', name='customer[email]', id='e', autocomplete='email', label='Email address', options=None),
    dict(i=2, tag='input', type='text', name='firstName', id='', autocomplete='off', label='', options=None),
    dict(i=3, tag='input', type='text', name='ship[ln]', id='', autocomplete='family-name', label='Family name', options=None),
    dict(i=4, tag='input', type='tel', name='contact', id='', autocomplete='tel', label='Email or phone', options=None),
    dict(i=5, tag='input', type='text', name='address1', id='', autocomplete=None, label='Address', options=None),
    dict(i=6, tag='input', type='text', name='address2', id='', autocomplete=None, label='Address line 2', options=None),
    dict(i=7, tag='input', type='text', name='locality', id='', autocomplete='address-level2', label='Suburb', options=None),
    dict(i=8, tag='select', type='select-one', name='region', id='', autocomplete='address-level1', label='State / Territory',
         options=[dict(v='VIC', t='Victoria'), dict(v='NSW', t='New South Wales')]),
    dict(i=9, tag='input', type='text', name='zip', id='', autocomplete='postal-code', label='Postcode', options=None),
]
hit = match_fields(FORM, dict(email='a@b.co', first_name='Sam', last_name='Nguyen', phone='0400 000 111',
                              address1='12 Smith St', address2='Unit 3', city='Brunswick',
                              state='Victoria', postcode='3056'))
assert {k: v['i'] for k, v in hit.items()} == {'email': 1, 'first_name': 2, 'last_name': 3, 'phone': 4,
                                              'address1': 5, 'address2': 6, 'city': 7, 'state': 8, 'postcode': 9}
assert hit['city']['label'] == 'Suburb'          # not "City", and found by autocomplete token
assert hit['first_name']['name'] == 'firstName'  # autocomplete="off" -> fall back to the name

# a field's own autocomplete token wins over a misleading label: i=4 says tel, so email skips it
assert _score(FORM[4], 'email') == 0 and _score(FORM[4], 'phone') == 100
assert _score(FORM[0], 'first_name') == 0        # the search box matches nothing
# one field is never claimed twice, and unmatched keys are reported rather than silently dropped
assert len({v['i'] for v in hit.values()}) == len(hit)
assert match_fields(FORM, dict(card_cvc='123')) == {}

In [ ]:
#: verification: only a real change counts, and only in a signal the page actually exposes
assert _changed(dict(count=0, subtotal=0), dict(count=1, subtotal=9.5)) == 'count'
assert _changed(dict(count=None, subtotal=10.0), dict(count=None, subtotal=19.5)) == 'subtotal'
assert _changed(dict(count=1, subtotal=9.5), dict(count=1, subtotal=9.5)) == ''
assert _changed(dict(count=1), dict(count=None)) == ''     # a reading that went blank is not progress
assert _changed(dict(count=1, lines=[{'title': 'a', 'qty': 1}]),
                dict(count=1, lines=[{'title': 'a', 'qty': 1}, {'title': 'b', 'qty': 1}])) == 'lines'

In [ ]:
#: picking a product never guesses — an unusable request raises, and says what was on offer
ITEMS = [dict(i=0, title='Oil Filter Z145A', related=None), dict(i=1, title='Oil Filter Z411', related=None),
         dict(i=2, title='Oil Filter Z411 Twin Pack', related=None), dict(i=3, title='Oil Filter Z145A', related=True)]
assert _match(ITEMS, 1)['i'] == 1                          # by index
assert _match(ITEMS, '1')['i'] == 1                        # ...including a stringified one
assert _match(ITEMS, 'Oil Filter Z411')['i'] == 1          # exact title beats the longer superstring
assert _match(ITEMS, 'twin pack')['i'] == 2                # substring, case-insensitive
assert _match(ITEMS, 'Oil Filter Z145A')['i'] == 0         # the real product, not the recommendation
for bad in ('flux capacitor', 9):
    try:
        _match(ITEMS, bad)
        raise AssertionError(f'{bad!r} should not have matched')
    except ShopError as e: assert 'Z145A' in str(e) or 'page has 4' in str(e), e

In [ ]:
#: the payment guard covers the wording checkouts actually use
for label in ('Pay now', 'Place order', 'Complete order', 'Confirm purchase', 'Buy now', 'Submit order'):
    assert PAY_RX.search(label), label
for label in ('Continue to shipping', 'Save address', 'Apply discount', 'Update cart'):
    assert not PAY_RX.search(label), label

In [ ]:
#: line lookup and numeric fill keys — both name what is really there rather than guessing
CART = dict(count=2, subtotal=13.0, lines=[dict(i=0, text='Apples', qty='2', qty_kind='select', remove=True),
                                           dict(i=1, text='Pears', qty='1', qty_kind=None, remove=True)])
assert _line(CART, 1)['text'] == 'Pears' and _line(CART, '0')['qty_kind'] == 'select'
try:
    _line(CART, 5)
    raise AssertionError('line 5 should not have resolved')
except ShopError as e: assert 'cart page shows 2' in str(e), e

#: a numeric profile key addresses a fields() index directly, so it never goes through FIELD_MAP
assert match_fields(FORM, {'7': 'Large', 'city': 'Brunswick'})['city']['i'] == 7   # '7' is not a profile key
assert _origin('https://shop.example.com/collections/all?x=1') == 'https://shop.example.com'

### Against live stores

Three storefronts with nothing in common: a Shopify store (deterministic `/cart.js` path), Amazon
(DOM clicks, verified through the header badge) and CERES Fair Food's Rails members shop (custom
markup, `<input type=submit value=Add>`, a quantity `<select>`, and no cart badge at all when you
are logged out — where `ok=None` is the honest answer).

In [ ]:
#| eval: false
s = shop('https://www.allbirds.com')
s.search('tree runner')[:2]

In [ ]:
#| eval: false
# no size chosen and several in stock: it asks instead of picking one
r = s.add(0)
r.get('need'), [v['name'] for v in (r.get('variants') or [])][:3]

In [ ]:
#| eval: false
s.add(0, qty=2, variant='9')        # -> {'ok': True, 'how': 'count', 'variant_id': '331912...', ...}

In [ ]:
#| eval: false
s.cart(lines=True), s.set_qty(0, 1), s.remove(0)

In [ ]:
#| eval: false
# CERES Fair Food: custom Rails storefront, "Add" buttons, quantity selects
c = shop('https://members.ceresfairfood.org.au')
[(p['title'], p['price'], p['qty']) for p in c.search('apples')[:3]]

In [ ]:
#| eval: false
# logged out there is no cart signal to check, so the result says so rather than claiming success
c.add('Apples Fuji Organic 500g', qty=2)   # -> {'ok': None, 'error': 'no cart signal ...', 'blockers': [...]}

## What this will not do

- **It will not pay.** `shop_fill(submit=...)` refuses anything matching `PAY_RX` unless you pass
  `confirm=True`, and nothing else in the module clicks a checkout button.
- **It will not beat a bot wall.** Coles serves an empty JS shell and Woolworths answers 403 to a
  datacentre IP; both work from a debug Chrome on a real machine, which is the point of running on
  the persistent browser rather than a fresh headless one.
- **It will not solve a captcha, log in, or choose your delivery store.** `shop_blockers()` names
  what is in the way so a human can do that part once — the session keeps it afterwards.
- **It will not claim success it cannot see.** `ok=None` is a real outcome and means "this page
  exposes no cart signal to verify against", not "probably fine".